In [1]:
print("jfngsgF")

jfngsgF


In [2]:
%pwd
import os
os.chdir("../")
#Just went one step bcakword with this directory....

%pwd

'd:\\mlops\\Crypto_Guardian'

In [44]:
import pandas as pd

df = pd.read_csv("artifacts\data_cleaning\data.csv")

df.head()

,Unnamed: 0,text,Label,Score
0,0,super kangaroo jacked,NEGATIVE,0.990524
1,1,arkansas don39t play man,NEGATIVE,0.976599
2,2,kangaroo listens nba young boy,POSITIVE,0.968622
3,3,whos watching 2001,POSITIVE,0.996961
4,4,ray making new show youtube made recap 3 video...,NEGATIVE,0.981149


In [45]:
df.columns

Index(['Unnamed: 0', 'text', 'Label', 'Score'], dtype='object')

In [13]:
df.drop(columns=[ 'Unnamed: 0',],inplace=True)

In [14]:
df.head()

,text,Label,Score
0,super kangaroo jacked,NEGATIVE,0.990524
1,arkansas don39t play man,NEGATIVE,0.976599
2,kangaroo listens nba young boy,POSITIVE,0.968622
3,whos watching 2001,POSITIVE,0.996961
4,ray making new show youtube made recap 3 video...,NEGATIVE,0.981149


In [16]:
nall = df.isnull().sum()

In [ ]:
def remove_null():
    nall = df.isnull().sum()
    
    if nall['text'] >0 :
        df.dropna(inplace=True)
    elif nall['Label'] >0:
        df.dropna(inplace=True)
    else:
        df.dropna(inplace=True)


In [24]:
df.isnull().sum()

text     0
Label    0
Score    0
dtype: int64

In [26]:
def remove_duplicate():
    if df.duplicated().sum() >0:
        df.drop_duplicates(inplace=True)

np.int64(75)

In [28]:
from sklearn.model_selection import train_test_split

In [30]:
train,test = train_test_split(df,test_size=0.25,random_state=42)

In [32]:
train.to_csv("train.csv")
test.to_csv("test.csv")

In [3]:

from dataclasses import dataclass
from pathlib import Path

@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    vectorizer_path: Path


In [4]:
from src.Crypto.constants import *
from src.Crypto.utils.helper import read_yaml,create_directories
from src.Crypto import logger

class ConfigurationManager:
    def __init__(self,
                 config_filepath=CONFIG_FILE_PATH,
                 params_filepath=CONFIG_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH
                 ):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)
        
        # create_directories([self.config.artifacts_root])
        
    def get_data_transformation_config(self)-> DataTransformationConfig:
        config =  self.config.data_transformation
        create_directories([config.root_dir])
        data_transformation_config  = DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path,
            vectorizer_path= config.vectorizer_path

        )
        return data_transformation_config


In [ ]:
from src.Crypto import logger
import pandas as pd
from sklearn.model_selection import train_test_split


class DataTransformationComponent:
    def __init__(self,config:DataTransformationConfig):
        self.config = config
        self.df = pd.read_csv(self.config.data_path)
        self.df.drop(columns=[ 'Unnamed: 0',],inplace=True)
    def remove_null(self):
        nall = self.df.isnull().sum()
        
        if nall['text'] >0 :
            self.df.dropna(inplace=True)
        elif nall['Label'] >0:
            self.df.dropna(inplace=True)
        else:
            self.df.dropna(inplace=True)
            
    def remove_duplicate(self):
        if self.df.duplicated().sum() >0:
           self.df.drop_duplicates(inplace=True)
    
    def split_data(self):
        train,test = train_test_split(df,test_size=0.25,random_state=42)
        
    def save_data(self):
        train.to_csv(os.path.join(self.config.root_dir,'train.csv'),index=False)
        test.to_csv(os.path.join(self.config.root_dir,'test.csv'),index=False)
        
    def transformation(self):
        self.remove_null()
        self.remove_duplicate()
        self.split_data()
        self.save_data()
        print("All Done...✅")
        
        


In [7]:
import os
from src.Crypto import logger
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sentence_transformers import SentenceTransformer
import numpy as np


class DataTransformationComponent:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.df = pd.read_csv(self.config.data_path)
        self.model = SentenceTransformer('all-mpnet-base-v2')
        # Check if column exists before dropping to avoid error
        if 'Unnamed: 0' in self.df.columns:
            self.df.drop(columns=['Unnamed: 0'], inplace=True)
            
    def remove_null(self):
        # Seedha dropna bhi kar sakte ho, agar null hai toh hat jayenge
        initial_shape = self.df.shape[0]
        self.df.dropna(inplace=True)
        new_shape = self.df.shape[0]
        logger.info(f"Removed {initial_shape - new_shape} null rows.")
            
    def remove_duplicate(self):
        if self.df.duplicated().sum() > 0:
           self.df.drop_duplicates(inplace=True)
           logger.info("Duplicates removed.")
           
    def label_encode(self):
        self.label_map = {"POSITIVE": 1, "NEGATIVE": 0}
        self.df['Label'] = self.df['Label'].map(self.label_map)

        
    def split_data(self):
        # self.train aur self.test banaya taaki poori class mein use ho sake
        self.train, self.test = train_test_split(
            self.df, 
            test_size=0.25, 
            random_state=42,
            stratify=self.df['Label'] # Recommendation for classification
        )
        logger.info(f"Data split into train ({self.train.shape}) and test ({self.test.shape})")
        
    def save_data(self):
        # Directory check
        os.makedirs(self.config.root_dir, exist_ok=True)
        
        # self.train aur self.test use kiya
        train_path = os.path.join(self.config.root_dir, 'train.csv')
        test_path = os.path.join(self.config.root_dir, 'test.csv')
        
        self.train.to_csv(train_path, index=False)
        self.test.to_csv(test_path, index=False)
        logger.info(f"Files saved at: {self.config.root_dir}")
     
    
        
    def text_encode(self):
        self.train_df = pd.read_csv(r"artifacts\data_transformation\train.csv")
        self.test_df = pd.read_csv(r"artifacts\data_transformation\test.csv")
        train_embeddings = self.model.encode(self.train_df['text'].tolist(),show_progress_bar=True)
        test_embeddings = self.model.encode(self.test_df['text'].tolist(),show_progress_bar=True)
        
        create_directories(["artifacts/embeddings"])
        joblib.dump(self.model, self.config.vectorizer_path)
        logger.debug(f"Model saved successfully! Download  from the files tab.")
        np.save(os.path.join("artifacts/embeddings", "train_embeddings.npy"), train_embeddings)
        np.save(os.path.join("artifacts/embeddings", "test_embeddings.npy"), test_embeddings)
        logger.info(f"Embeddings and Model saved successfully!")
      
    def transformation(self):
        # Saare steps sequence mein
        self.remove_null()
        self.remove_duplicate()
        self.label_encode()
        self.split_data()
        self.save_data()
        self.text_encode() # <--- Isse tabhi chalayein agar aapko embeddings save karni ho

        print("All Done...✅")

In [8]:
try:
    cfm = ConfigurationManager()
    data_transformation_config_manager = cfm.get_data_transformation_config()
    data_transformation_component = DataTransformationComponent(data_transformation_config_manager)
    data_transformation_component.transformation()
    logger.info("Pipeline Ran Sucesssfully ✅😭 ")
except Exception as e:
    logger.error("Are bhia kuch Error Aaa Gaya hai Yaar....")
    raise e

[06-05-2026 01:41:18 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[06-05-2026 01:41:18 PM - helper - INFO - Yaml File :config\config.yaml Read Sucessfully✅]
[06-05-2026 01:41:18 PM - helper - INFO - Yaml File :schema.yaml Read Sucessfully✅]
[06-05-2026 01:41:18 PM - helper - INFO - Folder Created Sucessfully: artifacts/data_transformation]
[06-05-2026 01:41:18 PM - model - INFO - No device provided, using cpu]
[06-05-2026 01:41:19 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"]


[06-05-2026 01:41:19 PM - _http - WARNING - Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[06-05-2026 01:41:19 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/modules.json "HTTP/1.1 200 OK"]
[06-05-2026 01:41:19 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"]
[06-05-2026 01:41:19 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-mpnet-base-v2/e8c3b32edf5434bc2275fc9bab85f82640a19130/config_sentence_transformers.json "HTTP/1.1 200 OK"]
[06-05-2026 01:41:19 PM - model - INFO - Loading SentenceTransformer model from sentence-transformers/all-mpnet-base-v2.]
[06-05-2026 01:41:20 

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3352.92it/s]


[06-05-2026 01:41:23 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"]
[06-05-2026 01:41:23 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[06-05-2026 01:41:23 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"]
[06-05-2026 01:41:24 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"]
[06-05-2026 01:41:24 PM - _client - INFO - HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-mpnet-base-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[06-05-2026 01:41:24 PM - _client - INFO - HTT

Batches: 100%|██████████| 16/16 [00:06<00:00,  2.30it/s]

[06-05-2026 01:41:54 PM - helper - INFO - Folder Created Sucessfully: artifacts/embeddings]


[06-05-2026 01:41:55 PM - 745094079 - INFO - Embeddings and Model saved successfully!]
All Done...✅
[06-05-2026 01:41:55 PM - 1281826165 - INFO - Pipeline Ran Sucesssfully ✅😭 ]
